In [3]:
# ============================================================
# RGB-NIR PAIRING MANIFEST CREATION - ORDER-BASED PAIRING
# ============================================================
# Pairing rule:
# For each (distance, gesture) group:
#   1. Sort RGB videos by extracted serial/timestamp
#   2. Sort NIR videos by extracted serial/timestamp
#   3. Pair index-wise:
#        smallest RGB <-> smallest NIR
#        second RGB   <-> second NIR
#        ...
#
# This replaces the earlier incorrect cartesian-product pairing.
#
# Output:
# data/processed/manifests/paired_manifest.csv
# ============================================================

import re
import json
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ============================================================
# 1. PATHS (FOR NOTEBOOKS)
# ============================================================
cwd = Path.cwd()

if cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

RGB_ROOT = ROOT / "data" / "raw" / "RGB"
NIR_ROOT = ROOT / "data" / "raw" / "NIR"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".MP4", ".AVI", ".MOV", ".MKV"}

print("Current working directory :", cwd)
print("Project ROOT              :", ROOT)
print("RGB_ROOT                  :", RGB_ROOT)
print("NIR_ROOT                  :", NIR_ROOT)

# ============================================================
# 2. LIST VIDEOS
# ============================================================
def list_videos(root: Path):
    files = []
    if not root.exists():
        return files
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in VIDEO_EXTS:
            files.append(p)
    return sorted(files)

rgb_videos = list_videos(RGB_ROOT)
nir_videos = list_videos(NIR_ROOT)

print(f"\nRGB videos found : {len(rgb_videos)}")
print(f"NIR videos found : {len(nir_videos)}")

if len(rgb_videos) == 0 or len(nir_videos) == 0:
    raise FileNotFoundError(
        f"No videos found.\nChecked:\nRGB -> {RGB_ROOT}\nNIR -> {NIR_ROOT}\nMake sure videos are extracted there."
    )

print("\nSample RGB paths:")
for p in rgb_videos[:5]:
    print(" ", p)

print("\nSample NIR paths:")
for p in nir_videos[:5]:
    print(" ", p)

# ============================================================
# 3. ALLOWED GESTURES
# ============================================================
ALLOWED_GESTURES = {
    "help",
    "doctor",
    "stand_up",
    "sit_down",
    "fire",
    "robbery",
    "emergency",
}

GESTURE_ALIASES = {
    "help": "help",
    "doctor": "doctor",
    "stand_up": "stand_up",
    "standup": "stand_up",
    "stand-up": "stand_up",
    "sit_down": "sit_down",
    "sitdown": "sit_down",
    "sit-down": "sit_down",
    "fire": "fire",
    "robbery": "robbery",
    "emergency": "emergency",
}

DISTANCE_PATTERNS = [
    r"(\d+)\s*[_-]?\s*feet",
    r"(\d+)\s*[_-]?\s*ft",
    r"(\d+)\s*feet",
    r"(\d+)\s*ft",
]

# ============================================================
# 4. HELPERS
# ============================================================
def normalize_text(x: str):
    return x.strip().lower().replace(" ", "_").replace("-", "_")

def normalize_gesture(name: str):
    x = normalize_text(name)
    return GESTURE_ALIASES.get(x, None)

def extract_distance(text: str):
    txt = normalize_text(text)
    for pat in DISTANCE_PATTERNS:
        m = re.search(pat, txt)
        if m:
            return f"{m.group(1)}_feet"
    return None

def extract_gesture_from_parts(parts):
    for part in parts:
        g = normalize_gesture(part)
        if g in ALLOWED_GESTURES:
            return g
    return None

def extract_numeric_serial(filename: str):
    """
    Extract the longest digit sequence from filename stem.
    RGB example:
        video_20260303_154701.mp4 -> 20260303154701
    NIR example:
        20260303154701830.mp4 -> 20260303154701830

    We use this only for sorting inside each group.
    """
    stem = Path(filename).stem
    digit_groups = re.findall(r"\d+", stem)
    if not digit_groups:
        return None

    # Join all digit groups to preserve RGB date+time pattern
    serial = "".join(digit_groups)
    try:
        return int(serial)
    except:
        return None

def parse_metadata(video_path: Path, modality: str):
    distance = None
    for part in video_path.parts:
        distance = extract_distance(part)
        if distance:
            break

    gesture = extract_gesture_from_parts(video_path.parts)
    filename = video_path.name
    file_stem = video_path.stem
    serial_no = extract_numeric_serial(filename)

    return {
        "modality": modality,
        "distance": distance,
        "gesture": gesture,
        "filename": filename,
        "file_stem": file_stem,
        "serial_no": serial_no,
        "video_path": str(video_path),
    }

def get_video_info(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return {
            "fps": np.nan,
            "num_frames": np.nan,
            "duration_sec": np.nan,
            "width": np.nan,
            "height": np.nan,
            "is_readable": False,
        }

    fps = cap.get(cv2.CAP_PROP_FPS)
    num_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    duration_sec = num_frames / fps if fps and fps > 0 else np.nan

    cap.release()

    return {
        "fps": float(fps) if fps is not None else np.nan,
        "num_frames": int(num_frames) if not np.isnan(num_frames) else np.nan,
        "duration_sec": float(duration_sec) if not np.isnan(duration_sec) else np.nan,
        "width": int(width) if not np.isnan(width) else np.nan,
        "height": int(height) if not np.isnan(height) else np.nan,
        "is_readable": True,
    }

# ============================================================
# 5. BUILD DATAFRAMES
# ============================================================
def build_df(video_paths, modality):
    rows = []
    for vp in tqdm(video_paths, desc=f"Processing {modality} videos"):
        meta = parse_metadata(vp, modality)
        info = get_video_info(vp)
        rows.append({**meta, **info})
    return pd.DataFrame(rows)

df_rgb = build_df(rgb_videos, "RGB")
df_nir = build_df(nir_videos, "NIR")

print("\nRGB dataframe shape:", df_rgb.shape)
print("NIR dataframe shape:", df_nir.shape)

# Keep only allowed gestures
df_rgb = df_rgb[df_rgb["gesture"].isin(ALLOWED_GESTURES)].copy()
df_nir = df_nir[df_nir["gesture"].isin(ALLOWED_GESTURES)].copy()

print("\nUnique gestures found in RGB:", sorted(df_rgb["gesture"].dropna().unique().tolist()))
print("Unique gestures found in NIR:", sorted(df_nir["gesture"].dropna().unique().tolist()))

# ============================================================
# 6. CHECK COUNTS PER (DISTANCE, GESTURE)
# ============================================================
rgb_group_counts = (
    df_rgb.groupby(["distance", "gesture"])
    .size()
    .reset_index(name="rgb_count")
)

nir_group_counts = (
    df_nir.groupby(["distance", "gesture"])
    .size()
    .reset_index(name="nir_count")
)

group_count_table = pd.merge(
    rgb_group_counts,
    nir_group_counts,
    on=["distance", "gesture"],
    how="outer"
).fillna(0)

group_count_table["rgb_count"] = group_count_table["rgb_count"].astype(int)
group_count_table["nir_count"] = group_count_table["nir_count"].astype(int)
group_count_table["pairable_count"] = group_count_table[["rgb_count", "nir_count"]].min(axis=1)

print("\n========== COUNT CHECK BY (DISTANCE, GESTURE) ==========")
display(group_count_table.sort_values(["distance", "gesture"]))

# ============================================================
# 7. SORT WITHIN EACH GROUP AND PAIR INDEX-WISE
# ============================================================
paired_rows = []
unpaired_summary = []

all_groups = sorted(set(zip(df_rgb["distance"], df_rgb["gesture"])) | set(zip(df_nir["distance"], df_nir["gesture"])))

pair_id_counter = 1

for distance, gesture in all_groups:
    rgb_group = df_rgb[(df_rgb["distance"] == distance) & (df_rgb["gesture"] == gesture)].copy()
    nir_group = df_nir[(df_nir["distance"] == distance) & (df_nir["gesture"] == gesture)].copy()

    # sort by serial_no, fallback by filename
    rgb_group = rgb_group.sort_values(["serial_no", "filename"], na_position="last").reset_index(drop=True)
    nir_group = nir_group.sort_values(["serial_no", "filename"], na_position="last").reset_index(drop=True)

    rgb_count = len(rgb_group)
    nir_count = len(nir_group)
    pair_count = min(rgb_count, nir_count)

    unpaired_summary.append({
        "distance": distance,
        "gesture": gesture,
        "rgb_count": rgb_count,
        "nir_count": nir_count,
        "paired_count": pair_count,
        "rgb_left_unpaired": rgb_count - pair_count,
        "nir_left_unpaired": nir_count - pair_count,
    })

    for i in range(pair_count):
        r = rgb_group.iloc[i]
        n = nir_group.iloc[i]

        fps_diff = abs(r["fps"] - n["fps"]) if pd.notna(r["fps"]) and pd.notna(n["fps"]) else np.nan
        frame_diff = abs(r["num_frames"] - n["num_frames"]) if pd.notna(r["num_frames"]) and pd.notna(n["num_frames"]) else np.nan
        duration_diff = abs(r["duration_sec"] - n["duration_sec"]) if pd.notna(r["duration_sec"]) and pd.notna(n["duration_sec"]) else np.nan

        is_valid_pair = (
            bool(r["is_readable"]) and
            bool(n["is_readable"]) and
            pd.notna(r["gesture"]) and
            pd.notna(n["gesture"]) and
            r["gesture"] == n["gesture"] and
            pd.notna(r["distance"]) and
            pd.notna(n["distance"]) and
            r["distance"] == n["distance"]
        )

        paired_rows.append({
            "pair_id": f"PAIR_{pair_id_counter:05d}",
            "pair_index_within_group": i + 1,
            "subject_id": None,  # not available from your current naming
            "gesture": gesture,
            "distance": distance,
            "capture_id": i + 1,  # sequential pairing index
            "rgb_video_path": r["video_path"],
            "nir_video_path": n["video_path"],
            "rgb_num_frames": r["num_frames"],
            "nir_num_frames": n["num_frames"],
            "rgb_fps": r["fps"],
            "nir_fps": n["fps"],
            "duration_rgb": r["duration_sec"],
            "duration_nir": n["duration_sec"],
            "fps_diff": fps_diff,
            "frame_diff": frame_diff,
            "duration_diff": duration_diff,
            "is_valid_pair": is_valid_pair,
            "rgb_width": r["width"],
            "rgb_height": r["height"],
            "nir_width": n["width"],
            "nir_height": n["height"],
            "rgb_filename": r["filename"],
            "nir_filename": n["filename"],
            "rgb_serial_no": r["serial_no"],
            "nir_serial_no": n["serial_no"],
        })

        pair_id_counter += 1

paired_manifest = pd.DataFrame(paired_rows)
unpaired_df = pd.DataFrame(unpaired_summary)

# ============================================================
# 8. SAVE FINAL CSV
# ============================================================
output_csv = MANIFEST_DIR / "paired_manifest.csv"
paired_manifest.to_csv(output_csv, index=False)

# also save group summary
group_summary_csv = MANIFEST_DIR / "pairing_group_summary.csv"
unpaired_df.to_csv(group_summary_csv, index=False)

print("\nSaved paired manifest to:")
print(output_csv)

print("\nSaved group summary to:")
print(group_summary_csv)

# ============================================================
# 9. SUMMARY
# ============================================================
summary = {
    "rgb_total_after_filter": int(len(df_rgb)),
    "nir_total_after_filter": int(len(df_nir)),
    "paired_total": int(len(paired_manifest)),
    "valid_pairs": int(paired_manifest["is_valid_pair"].sum()) if len(paired_manifest) else 0,
    "invalid_pairs": int((~paired_manifest["is_valid_pair"]).sum()) if len(paired_manifest) else 0,
    "unique_gestures_in_pairs": int(paired_manifest["gesture"].dropna().nunique()) if len(paired_manifest) else 0,
    "unique_distances_in_pairs": int(paired_manifest["distance"].dropna().nunique()) if len(paired_manifest) else 0,
    "gestures_in_pairs": sorted(paired_manifest["gesture"].dropna().unique().tolist()) if len(paired_manifest) else [],
}

print("\n========== FINAL ORDER-BASED PAIRING SUMMARY ==========")
print(json.dumps(summary, indent=4))

print("\nFirst 10 paired rows:")
display(paired_manifest.head(10))

print("\nGroup-wise pairing summary:")
display(unpaired_df.sort_values(["distance", "gesture"]))

print("\nGesture counts in paired manifest:")
display(paired_manifest["gesture"].value_counts().sort_index())

print("\nAny invalid pairs?")
display(paired_manifest[~paired_manifest["is_valid_pair"]].head(20))

Current working directory : /data/Sajjan_Singh/gesture_recognition/notebooks
Project ROOT              : /data/Sajjan_Singh/gesture_recognition
RGB_ROOT                  : /data/Sajjan_Singh/gesture_recognition/data/raw/RGB
NIR_ROOT                  : /data/Sajjan_Singh/gesture_recognition/data/raw/NIR

RGB videos found : 749
NIR videos found : 749

Sample RGB paths:
  /data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_154701.mp4
  /data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_155209.mp4
  /data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_160149.mp4
  /data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161125.mp4
  /data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161400.mp4

Sample NIR paths:
  /data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303154701830.mp4
  /data/Sajjan_Singh/gesture_recognition/data/r

Processing NIR videos: 100%|██████████| 749/749 [00:03<00:00, 191.43it/s]


RGB dataframe shape: (749, 13)
NIR dataframe shape: (749, 13)

Unique gestures found in RGB: ['doctor', 'emergency', 'fire', 'help', 'robbery', 'sit_down', 'stand_up']
Unique gestures found in NIR: ['doctor', 'emergency', 'fire', 'help', 'robbery', 'sit_down', 'stand_up']

========== COUNT CHECK BY (DISTANCE, GESTURE) ==========


,distance,gesture,rgb_count,nir_count,pairable_count
0,4_feet,doctor,34,34,34
1,4_feet,emergency,34,34,34
2,4_feet,fire,34,34,34
3,4_feet,help,34,34,34
4,4_feet,robbery,34,34,34
5,4_feet,sit_down,34,34,34
6,4_feet,stand_up,34,34,34
7,6_feet,doctor,35,35,35
8,6_feet,emergency,35,35,35
9,6_feet,fire,35,35,35



Saved paired manifest to:
/data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest.csv

Saved group summary to:
/data/Sajjan_Singh/gesture_recognition/data/processed/manifests/pairing_group_summary.csv

========== FINAL ORDER-BASED PAIRING SUMMARY ==========
{
    "rgb_total_after_filter": 749,
    "nir_total_after_filter": 749,
    "paired_total": 749,
    "valid_pairs": 749,
    "invalid_pairs": 0,
    "unique_gestures_in_pairs": 7,
    "unique_distances_in_pairs": 3,
    "gestures_in_pairs": [
        "doctor",
        "emergency",
        "fire",
        "help",
        "robbery",
        "sit_down",
        "stand_up"
    ]
}

First 10 paired rows:


...

,distance,gesture,rgb_count,nir_count,paired_count,rgb_left_unpaired,nir_left_unpaired
0,4_feet,doctor,34,34,34,0,0
1,4_feet,emergency,34,34,34,0,0
2,4_feet,fire,34,34,34,0,0
3,4_feet,help,34,34,34,0,0
4,4_feet,robbery,34,34,34,0,0
5,4_feet,sit_down,34,34,34,0,0
6,4_feet,stand_up,34,34,34,0,0
7,6_feet,doctor,35,35,35,0,0
8,6_feet,emergency,35,35,35,0,0
9,6_feet,fire,35,35,35,0,0



Gesture counts in paired manifest:


gesture
doctor       107
emergency    107
fire         107
help         107
robbery      107
sit_down     107
stand_up     107
Name: count, dtype: int64


Any invalid pairs?


,pair_id,pair_index_within_group,subject_id,gesture,distance,capture_id,rgb_video_path,nir_video_path,rgb_num_frames,nir_num_frames,rgb_fps,nir_fps,duration_rgb,duration_nir,fps_diff,frame_diff,duration_diff,is_valid_pair,rgb_width,rgb_height,nir_width,nir_height,rgb_filename,nir_filename,rgb_serial_no,nir_serial_no
